# BIOBERT: WordPiece Tokenization for NER Task

In [9]:
from transformers import AutoTokenizer, AutoModelForTokenClassification
import json

# NOTE:
#[https://github.com/huggingface/transformers/blob/v5.0.0rc0/src/transformers/models/distilbert/tokenization_distilbert.py#L23]
# it appears that DistilBertTokenizerFast, DistilBertTokenizer are now aliases

# ========================
# LOAD TOKENIZER AND MODEL
# ========================
MODEL_NAME = "dmis-lab/biobert-base-cased-v1.1"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForTokenClassification.from_pretrained(MODEL_NAME)

dataset = []
with open('data/synthetic_data_tokenized.jsonl', 'r') as f:
    for line in f:
        row = json.loads(line)
        dataset.append(json.loads(line))


Some weights of BertForTokenClassification were not initialized from the model checkpoint at dmis-lab/biobert-base-cased-v1.1 and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [10]:
print("Model hidden size: ", model.config.hidden_size)

Model hidden size:  768


In [11]:
# check that the labels appear as expected
dataset[-2:]

[{'text': 'Patient notes junctional tachycardia.',
  'word_tokens': ['Patient', 'notes', 'junctional', 'tachycardia', '.'],
  'word_labels': ['O', 'O', 'B-SYMPTOM_POS', 'I-SYMPTOM_POS', 'O'],
  'symptom_id': 's0202',
  'is_negated': False},
 {'text': 'Not experiencing skin desquamation.',
  'word_tokens': ['Not', 'experiencing', 'skin', 'desquamation', '.'],
  'word_labels': ['O', 'O', 'B-SYMPTOM_NEG', 'I-SYMPTOM_NEG', 'O'],
  'symptom_id': 's0297',
  'is_negated': True}]

# Tokenize Dataset using BioBERT's Tokenizer

Convert pre-tokenized whitespace tokens + BIO labels into model (wordpiece) tokens and produce aligned label ids per model token

## STEPS:
1. Give the white space tokens to the tokenizer
        - This will allow the proper assignment of the Labels to the subword tokens

2. Label each token (WordPiece tokenized) based on the words that these tokens belong to.

3. Convert the string labels to integer labels

### Quick Look at How it Works

In [12]:
# Load all data from JSONL into a list of dictionaries
ex1 = dataset[0]
print(ex1.keys())
print("TOkenize text")
toks = tokenizer(ex1['text'])
print(toks)

print("\nTokenize tokens")

print("with is_split_into_words=False")
toks_ids = tokenizer(ex1['word_tokens'],is_split_into_words=False) #?? DEFAULT = FALSE
print(toks_ids)
# Will not work because of nested lists
#print("Tokens:\n", tokenizer.convert_ids_to_tokens(toks_ids['input_ids']))

print("\nwith is_split_into_words=True")
toks_ids = tokenizer(ex1['word_tokens'],is_split_into_words=True) #??
print(toks_ids)
print("Word ids:\n", toks_ids.word_ids())
print("Tokens:\n", tokenizer.convert_ids_to_tokens(toks_ids['input_ids']))

# [CLS] -> Classification token (start of sequence)
# [SEP] -> Separator token (end of sequence)

# Tokenization flow: words -> word ids -> token ids

dict_keys(['text', 'word_tokens', 'word_labels', 'symptom_id', 'is_negated'])
TOkenize text
{'input_ids': [101, 1175, 1132, 1185, 8006, 1104, 188, 19091, 8380, 119, 102], 'token_type_ids': [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], 'attention_mask': [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]}

Tokenize tokens
with is_split_into_words=False
{'input_ids': [[101, 1175, 102], [101, 1132, 102], [101, 1185, 102], [101, 8006, 102], [101, 1104, 102], [101, 188, 19091, 8380, 102], [101, 119, 102]], 'token_type_ids': [[0, 0, 0], [0, 0, 0], [0, 0, 0], [0, 0, 0], [0, 0, 0], [0, 0, 0, 0, 0], [0, 0, 0]], 'attention_mask': [[1, 1, 1], [1, 1, 1], [1, 1, 1], [1, 1, 1], [1, 1, 1], [1, 1, 1, 1, 1], [1, 1, 1]]}

with is_split_into_words=True
{'input_ids': [101, 1175, 1132, 1185, 8006, 1104, 188, 19091, 8380, 119, 102], 'token_type_ids': [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], 'attention_mask': [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]}
Word ids:
 [None, 0, 1, 2, 3, 4, 5, 5, 5, 6, None]
Tokens:
 ['[CLS]', 'there', 'are', 'no', 'symptom

# Tokenization

- assign -100 to special tokens: [CLS], [SEP], model does not need to learn the labels for this special tokens

Sources:
[]
[https://medium.com/@whyamit101/fine-tuning-bert-for-named-entity-recognition-ner-b42bcf55b51d]

In [13]:
ex = dataset[10]
# is_split_into_words=True: Input is pre-tokenized list of words, not a string
# - Tokenizer applies subword tokenization to each word individually
# - Returns word_ids() mapping each subword token back to its original word index
# - Critical for token classification: aligns word-level labels with subword tokens
# - Without it: tokenizer treats list as nested structure, causing errors
toks_ids = tokenizer(ex['word_tokens'],is_split_into_words=True)
print("Text:\n\t", ex['text'])
print("Input token ids:\n\t",toks_ids['input_ids'])
print("Word ids:\n\t", toks_ids.word_ids())
print("Tokens:\n\t", tokenizer.convert_ids_to_tokens(toks_ids['input_ids']))


Text:
	 Reports periumbilic pelvic lump.
Input token ids:
	 [101, 3756, 1679, 3656, 15197, 1596, 185, 1883, 15901, 16401, 119, 102]
Word ids:
	 [None, 0, 1, 1, 1, 1, 2, 2, 2, 3, 4, None]
Tokens:
	 ['[CLS]', 'reports', 'per', '##ium', '##bil', '##ic', 'p', '##el', '##vic', 'lump', '.', '[SEP]']


In [14]:
# ===========================================================================
# Create wordpiece-tokenized dataset 
# ===========================================================================

import json

# Prepare list for storing tokenized samples (optional, for downstream use)
tokenized_samples = []

for row in dataset:

    # Tokenize word tokens for each sample
    words = row['word_tokens']
    toks_ids = tokenizer(words, is_split_into_words=True)
    input_ids = toks_ids['input_ids']
    word_ids = toks_ids.word_ids()

    tokens = tokenizer.convert_ids_to_tokens(input_ids)
    token_labels = []
    previous_word_id = None

    for i, tok in enumerate(input_ids):
        word_id = word_ids[i]

        # Special tokens
        if word_id is None:
            token_labels.append("None") # -100
        # First subword of a word
        elif word_id != previous_word_id:
            token_labels.append(row['word_labels'][word_id])
        # Continuation subwords
        else:
            original_label = row['word_labels'][word_id]
            # If it's a B- tag, convert it to I-
            if isinstance(original_label, str) and original_label.startswith("B-"):
                fixed_label = original_label.replace("B-", "I-")
            else:
                fixed_label = original_label
            token_labels.append(fixed_label)
        previous_word_id = word_id

    # Prepare record for saving
    tokenized_sample = {
        "text": row["text"],
        "word_tokens": words,
        "word_labels": row["word_labels"],
        "tokens": tokens,
        "input_ids": input_ids,
        "token_labels": token_labels
    }
    tokenized_samples.append(tokenized_sample)

print("Example tokenized sample:")
print(tokenized_samples[0])


Example tokenized sample:
{'text': 'There are no symptoms of stridor.', 'word_tokens': ['There', 'are', 'no', 'symptoms', 'of', 'stridor', '.'], 'word_labels': ['O', 'O', 'O', 'O', 'O', 'B-SYMPTOM_NEG', 'O'], 'tokens': ['[CLS]', 'there', 'are', 'no', 'symptoms', 'of', 's', '##tri', '##dor', '.', '[SEP]'], 'input_ids': [101, 1175, 1132, 1185, 8006, 1104, 188, 19091, 8380, 119, 102], 'token_labels': ['None', 'O', 'O', 'O', 'O', 'O', 'B-SYMPTOM_NEG', 'I-SYMPTOM_NEG', 'I-SYMPTOM_NEG', 'O', 'None']}


In [15]:
with open("data/data_wordpiece_tokenized_biobert.jsonl", "w") as f:
    for sample in tokenized_samples:
        f.write(json.dumps(sample) + "\n")

## Convert String Labels to Integer Labels

In [16]:
import json
dataset = []
with open("data/data_wordpiece_tokenized_biobert.jsonl", "r") as f:
    for line in f:
        dataset.append(json.loads(line))

dataset[-1]

{'text': 'Not experiencing skin desquamation.',
 'word_tokens': ['Not', 'experiencing', 'skin', 'desquamation', '.'],
 'word_labels': ['O', 'O', 'B-SYMPTOM_NEG', 'I-SYMPTOM_NEG', 'O'],
 'tokens': ['[CLS]',
  'not',
  'experiencing',
  'skin',
  'des',
  '##qua',
  '##mation',
  '.',
  '[SEP]'],
 'input_ids': [101, 1136, 13992, 2241, 3532, 13284, 16059, 119, 102],
 'token_labels': ['None',
  'O',
  'O',
  'B-SYMPTOM_NEG',
  'I-SYMPTOM_NEG',
  'I-SYMPTOM_NEG',
  'I-SYMPTOM_NEG',
  'O',
  'None']}

In [ ]:
# Collect all unique labels from your dataset and saved it in jsons

unique_labels = set()

for row in dataset: 
    for lbl in row["token_labels"]:
        if lbl != "None":  # ignore special tokens
            unique_labels.add(lbl)

# Sort for stable ordering
unique_labels = sorted(list(unique_labels))

# Create mappings
label2id = {label: idx for idx, label in enumerate(unique_labels)}
id2label = {idx: label for label, idx in label2id.items()}

print("Number of labels:", len(label2id))

# Save mapping:

# THIS IS THE SAME AS FOR DISTILLBERT!
# SAME LABELS
# - symptoms: B/I , POS/NEG
# - O
# with open("data/label2id.json", "w") as f:
#     json.dump(label2id, f, indent = 2)
# with open("data/id2label.json", "w") as f:
#     json.dump(id2label, f, indent = 2)

Number of labels: 5


In [18]:
# Convert labels to numeric IDs
for row in dataset:
    row["token_label_ids"] = [
        -100 if lbl == "None" else label2id[lbl]
        for lbl in row["token_labels"]
    ]

with open("data/data_wordpiece_tokenized_biobert.jsonl", "w") as f:
    for sample in dataset:
        f.write(json.dumps(sample) + "\n")

In [19]:
# Sanity check!

In [22]:
for i in range(42):
    print(f"Example {i}:")
    print("token_labels:   ", dataset[i]["token_labels"])
    print("token_label_ids:", dataset[i]["token_label_ids"])
    print("-" * 40)

Example 0:
token_labels:    ['None', 'O', 'O', 'O', 'O', 'O', 'B-SYMPTOM_NEG', 'I-SYMPTOM_NEG', 'I-SYMPTOM_NEG', 'O', 'None']
token_label_ids: [-100, 4, 4, 4, 4, 4, 0, 2, 2, 4, -100]
----------------------------------------
Example 1:
token_labels:    ['None', 'O', 'O', 'O', 'B-SYMPTOM_POS', 'I-SYMPTOM_POS', 'O', 'None']
token_label_ids: [-100, 4, 4, 4, 1, 3, 4, -100]
----------------------------------------
Example 2:
token_labels:    ['None', 'O', 'B-SYMPTOM_POS', 'I-SYMPTOM_POS', 'O', 'None']
token_label_ids: [-100, 4, 1, 3, 4, -100]
----------------------------------------
Example 3:
token_labels:    ['None', 'O', 'O', 'B-SYMPTOM_POS', 'I-SYMPTOM_POS', 'I-SYMPTOM_POS', 'O', 'None']
token_label_ids: [-100, 4, 4, 1, 3, 3, 4, -100]
----------------------------------------
Example 4:
token_labels:    ['None', 'O', 'O', 'B-SYMPTOM_NEG', 'I-SYMPTOM_NEG', 'I-SYMPTOM_NEG', 'O', 'None']
token_label_ids: [-100, 4, 4, 0, 2, 2, 4, -100]
----------------------------------------
Example 5:
token